<a href="https://colab.research.google.com/github/NiceShot07/Recipe-Ingredient-Extractor/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recipe Ingredient Extractor

Proiect pentru materia Natural Language Processing and Text Mining.

## Scopul proiectului

Vrem să construim un program care extrage cantitatea, unitatea de
măsură și numele ingredientului din texte de rețete în limba engleză.

De exemplu, din textul `200 g flour` vrem să obținem:
- cantitate: `200`
- unitate de măsură: `g`
- ingredient: `flour`

Vom folosi Python, expresii regulate și un model de clasificare.

## Rolul acestui notebook

În acest notebook vom încărca și explora datele. Vom verifica
structura lor, valorile lipsă și câteva exemple de ingrediente.

Pentru început, folosim câteva exemple scrise de noi.
Datasetul real va fi adăugat într-o etapă următoare.

In [10]:
# Exemple scrise de noi pentru a verifica formatul datelor.
# Includem și cazuri fără unitate sau fără cantitate numerică.
ingredient_examples = [
    "200 g flour",
    "2 large eggs",
    "1 tablespoon olive oil",
    "1/2 cup milk",
    "salt to taste",
]

print(f"Număr de exemple: {len(ingredient_examples)}\n")

for ingredient in ingredient_examples:
    print(ingredient)

Număr de exemple: 5

200 g flour
2 large eggs
1 tablespoon olive oil
1/2 cup milk
salt to taste


## Organizarea exemplelor într-un tabel

Folosim biblioteca pandas pentru a organiza exemplele într-un tabel,
numit DataFrame. Fiecare rând conține textul unui ingredient.

Păstrăm textele originale pentru a putea verifica ulterior rezultatele
extragerii.

In [11]:
import pandas as pd

# Fiecare exemplu devine un rând în coloana ingredient_text.
examples_df = pd.DataFrame({
    "ingredient_text": ingredient_examples
})

display(examples_df)

,ingredient_text
0,200 g flour
1,2 large eggs
2,1 tablespoon olive oil
3,1/2 cup milk
4,salt to taste


## Datasetul real

Folosim datasetul publicat de The New York Times în proiectul
Ingredient Phrase Tagger.

Acesta conține texte de ingrediente în limba engleză și informații
asociate, precum cantitatea, unitatea de măsură și numele ingredientului.

Sursa: https://github.com/nytimes/ingredient-phrase-tagger

Încărcăm datele originale și verificăm primele rânduri.
În această etapă nu modificăm și nu eliminăm înregistrări.

In [12]:
# Citim fișierul CSV direct din repository-ul sursă.
dataset_url = (
    "https://raw.githubusercontent.com/nytimes/"
    "ingredient-phrase-tagger/master/nyt-ingredients-snapshot-2015.csv"
)

raw_df = pd.read_csv(dataset_url)

print(f"Număr de rânduri: {raw_df.shape[0]}")
print(f"Număr de coloane: {raw_df.shape[1]}")

display(raw_df.head())

Număr de rânduri: 179207
Număr de coloane: 7


,index,input,name,qty,range_end,unit,comment
0,0,1 1/4 cups cooked and pureed fresh butternut s...,butternut squash,1.25,0.0,cup,"cooked and pureed fresh, or 1 10-ounce package..."
1,1,1 cup peeled and cooked fresh chestnuts (about...,chestnuts,1.00,0.0,cup,"peeled and cooked fresh (about 20), or 1 cup c..."
2,2,"1 medium-size onion, peeled and chopped",onion,1.00,0.0,NaN,"medium-size, peeled and chopped"
3,3,"2 stalks celery, chopped coarse",celery,2.00,0.0,stalk,chopped coarse
4,4,1 1/2 tablespoons vegetable oil,vegetable oil,1.50,0.0,tablespoon,NaN


## Verificarea structurii și a valorilor lipsă

Datasetul conține câte o expresie de ingredient pe fiecare rând.
Coloana input reprezintă textul de intrare, iar celelalte câmpuri
conțin informațiile de referință.

Verificăm tipurile de date și numărul valorilor lipsă.
O unitate sau un comentariu lipsă poate fi normal, așa că nu
eliminăm automat rândurile cu astfel de valori.

In [13]:
# Verificăm ce tip de date conține fiecare coloană.
raw_df.info()

# Numărăm valorile lipsă și calculăm procentul pentru fiecare coloană.
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
})

display(missing_summary)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179207 entries, 0 to 179206
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   index      179207 non-null  int64  
 1   input      179063 non-null  object 
 2   name       178759 non-null  object 
 3   qty        179207 non-null  float64
 4   range_end  173986 non-null  float64
 5   unit       123082 non-null  object 
 6   comment    98658 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 9.6+ MB


,missing_count,missing_percent
index,0,0.00
input,144,0.08
name,448,0.25
qty,0,0.00
range_end,5221,2.91
unit,56125,31.32
comment,80549,44.95


## Observații inițiale

Datasetul conține 179.207 expresii de ingrediente.

Există 144 de valori lipsă în coloana input și 448 în coloana name.
Vom analiza aceste cazuri înainte de pregătirea datelor de lucru.

Lipsa unității sau a comentariului poate fi normală.
Păstrăm aceste exemple în această etapă.

În continuare, verificăm textele repetate. Acestea sunt importante
deoarece același text în antrenare și în test poate face evaluarea
prea optimistă.

In [14]:
# Ignorăm identificatorul când căutăm înregistrări identice.
content_columns = ["input", "name", "qty", "range_end", "unit", "comment"]

duplicate_records = raw_df.duplicated(subset=content_columns).sum()

# Excludem textele lipsă din verificarea expresiilor repetate.
input_counts = raw_df["input"].dropna().value_counts()
repeated_inputs = input_counts[input_counts > 1]

print(f"Înregistrări identice în plus: {duplicate_records}")
print(f"Texte distincte care apar de mai multe ori: {len(repeated_inputs)}")

display(repeated_inputs.head(10).rename("occurrences").to_frame())

Înregistrări identice în plus: 81200
Texte distincte care apar de mai multe ori: 14005


,occurrences
input,
Freshly ground pepper to taste,1473
Salt,1227
Salt to taste,1164
2 tablespoons olive oil,751
1/2 teaspoon salt,725
Salt and freshly ground pepper to taste,693
Salt and freshly ground black pepper,672
1 bay leaf,669
1 teaspoon salt,574


## Concluzii ale explorării inițiale

Am încărcat datasetul, care conține 179.207 rânduri și 7 coloane.

Am identificat:
- 144 de valori lipsă în coloana input;
- 448 de valori lipsă în coloana name;
- 81.200 de înregistrări duplicate, fără a lua în calcul indexul;
- 14.005 texte distincte care apar de mai multe ori.

Lipsa unei unități sau a unui comentariu poate fi normală,
deci nu vom elimina automat aceste exemple.

Datele originale nu au fost modificate. În etapa următoare vom
pregăti o copie de lucru și vom stabili cum tratăm datele lipsă
și duplicatele.

La împărțirea datelor, vom evita ca același text să apară
atât în setul de antrenare, cât și în cel de test.